# Lab 4: Deploy the Recommendation and Compare

> ⚠️ **Prerequisite**: This lab requires output from **Lab 1** (baseline metrics) and **Lab 3** (recommendation job results). If you haven't completed those labs, go back and run them first.

This is the culmination of the workshop. You will:

1. **Load** the best recommendation from Lab 3
2. **Deploy** the optimized configuration to a SageMaker endpoint
3. **Test** inference on the optimized endpoint
4. **Benchmark** using the same workload as Lab 1 (apples-to-apples comparison)
5. **Compare** baseline vs. optimized performance side-by-side


## Setup

In [ ]:
%pip install -r ../requirements.txt --quiet --no-warn-conflicts

In [ ]:
import json
import time
import os
import boto3
import pandas as pd
from datetime import datetime

import sys
sys.path.append("..")
from utils import (
    get_execution_role,
    wait_for_endpoint,
    wait_for_benchmark_job,
    parse_benchmark_results,
    extract_metrics,
    format_metrics_table,
    print_comparison_table,
    cleanup_resources,
)

# Resolve execution role
role = get_execution_role()
region = boto3.session.Session().region_name or "us-west-2"
account_id = boto3.client("sts").get_caller_identity()["Account"]
bucket = f"sagemaker-{region}-{account_id}"
timestamp = datetime.now().strftime("%m%d%H%M%S")

# Initialize clients
sm_client = boto3.client("sagemaker", region_name=region)
smr_client = boto3.client("sagemaker-runtime", region_name=region)
s3_client = boto3.client("s3", region_name=region)

print(f"Region:          {region}")
print(f"Role:            {role}")
print(f"Default bucket:  {bucket}")


## 1. Load the Recommendation from Lab 3

We saved the recommendation data in Lab 3. Let's load it and examine the best configuration.

In [ ]:
# Load recommendation data from Lab 3
with open("../results/lab3_recommendations.json", "r") as f:
    recommendation_data = json.load(f)

recommendations = recommendation_data["recommendations"]
recommendation_job_name = recommendation_data["job_name"]

print(f"Loaded {len(recommendations)} recommendations from job: {recommendation_job_name}")
print(f"\nSelecting Recommendation #1 (best for throughput)...")

# Select the top recommendation
best_rec = recommendations[0]
deploy_config = best_rec.get("DeploymentConfiguration", {})
model_details = best_rec.get("ModelDetails", {})
expected_performance = best_rec.get("ExpectedPerformance", [])
optimization_details = best_rec.get("OptimizationDetails", {})

print(f"\nDeployment Configuration:")
print(f"  Instance Type:     {deploy_config.get('InstanceType', 'N/A')}")
print(f"  Instance Count:    {deploy_config.get('InstanceCount', 1)}")
print(f"  Copies/Instance:   {deploy_config.get('CopyCountPerInstance', 1)}")
print(f"  Container:         {deploy_config.get('ImageUri', 'N/A')[:80]}...")
print(f"\nModel Package:       {model_details.get('ModelPackageArn', 'N/A')[:80]}")
print(f"Inference Spec:      {model_details.get('InferenceSpecificationName', 'N/A')}")
print(f"\nOptimizations Applied:")
if isinstance(optimization_details, dict):
    for technique, details in optimization_details.items():
        print(f"  - {technique}: {details}")
elif isinstance(optimization_details, list):
    for opt in optimization_details:
        print(f"  - {opt}")

## 2. Deploy the Recommended Configuration

The recommendation includes a **model package ARN** that bundles everything needed for deployment:
- Container image (optimized for the recommended instance type)
- Environment variables (tensor parallel degree, max sequence length, etc.)
- Model artifacts (including any optimization outputs like speculator weights)

This is the simplest deployment path - the model package already contains all the configuration.

In [ ]:
# Deploy the recommended configuration
optimized_endpoint_name = f"llama31-8b-optimized-{timestamp}"
optimized_model_name = f"llama31-8b-optimized-model-{timestamp}"
optimized_endpoint_config_name = f"llama31-8b-optimized-epc-{timestamp}"

# Step 1: Create model from the model package
# The model package contains the container, env vars, and S3 data channels
container_def = {
    "ModelPackageName": model_details["ModelPackageArn"],
}

# If a named inference specification is provided, include it
# (used when the model package has multiple variants for different optimizations)
if model_details.get("InferenceSpecificationName"):
    container_def["InferenceSpecificationName"] = model_details["InferenceSpecificationName"]

sm_client.create_model(
    ModelName=optimized_model_name,
    PrimaryContainer=container_def,
    ExecutionRoleArn=role,
)
print(f"Model created: {optimized_model_name}")

In [ ]:
# Step 2: Create endpoint configuration
instance_type = deploy_config["InstanceType"]
instance_count = deploy_config.get("InstanceCount", 1)
copy_count = deploy_config.get("CopyCountPerInstance", 1)

production_variant = {
    "VariantName": "AllTraffic",
    "ModelName": optimized_model_name,
    "InstanceType": instance_type,
    "InitialInstanceCount": instance_count,
}

# Handle multi-copy deployments
# When CopyCountPerInstance > 1, multiple copies of the model are loaded
# on each instance to increase throughput via request-level parallelism
if copy_count and copy_count > 1:
    production_variant["InferenceAmiVersion"] = "al2-ami-sagemaker-inference-gpu-2"
    production_variant["RoutingConfig"] = {"RoutingStrategy": "LEAST_OUTSTANDING_REQUESTS"}
    print(f"Multi-copy deployment: {copy_count} copies per instance")
    print(f"Routing strategy: LEAST_OUTSTANDING_REQUESTS")

sm_client.create_endpoint_config(
    EndpointConfigName=optimized_endpoint_config_name,
    ProductionVariants=[production_variant],
)
print(f"\nEndpoint config created: {optimized_endpoint_config_name}")
print(f"  Instance type: {instance_type}")
print(f"  Instance count: {instance_count}")

In [ ]:
# Step 3: Create the endpoint
sm_client.create_endpoint(
    EndpointName=optimized_endpoint_name,
    EndpointConfigName=optimized_endpoint_config_name,
)
print(f"Endpoint creation initiated: {optimized_endpoint_name}")
print(f"\nWaiting for endpoint to be InService (typically 8-15 minutes)...")

# Wait for endpoint to be ready
wait_for_endpoint(optimized_endpoint_name, region=region, timeout_minutes=30)

## 3. Test Inference on the Optimized Endpoint

Let's verify the optimized endpoint works correctly with the same test payload from Lab 1.

In [ ]:
# Test inference - same payload as Lab 1
payload = {
    "messages": [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "What is Amazon SageMaker AI? Answer in 2-3 sentences."}
    ],
    "max_tokens": 150,
    "temperature": 0.7,
    "stream": False,
}

response = smr_client.invoke_endpoint(
    EndpointName=optimized_endpoint_name,
    ContentType="application/json",
    Body=json.dumps(payload),
)

result = json.loads(response["Body"].read().decode("utf-8"))
print("Model response (optimized endpoint):")
print("-" * 40)
print(result["choices"][0]["message"]["content"])
print("-" * 40)
print(f"\nUsage: {result.get('usage', {})}")
print("\nThe optimized model produces the same quality output!")
print("Speculative decoding preserves the original model's distribution.")

## 4. Benchmark the Optimized Endpoint

Now we benchmark the optimized endpoint using the **exact same workload configuration** as Lab 1. This gives us an apples-to-apples comparison.

> 💡 **Key point**: Same model, same workload, same metrics - the only difference is the deployment configuration and optimizations applied by the recommendation service.

In [ ]:
# Create the same workload config as Lab 1 baseline
HF_TOKEN = "YOUR_HF_TOKEN_HERE"  # Replace with your token
secret_name = f"hf-token-compare-{timestamp}"

try:
    response = secretsmanager_client.create_secret(
        Name=secret_name,
        SecretString=HF_TOKEN,
        Description="HuggingFace token for comparison benchmark",
    )
    secret_arn = response["ARN"]
except secretsmanager_client.exceptions.ResourceExistsException:
    response = secretsmanager_client.describe_secret(SecretId=secret_name)
    secret_arn = response["ARN"]

# Same workload as Lab 1 - identical parameters for fair comparison
workload_config_name = f"bench-compare-{timestamp}"

workload_spec = {
    "benchmark": {"type": "aiperf"},
    "parameters": {
        "prompt_input_tokens_mean": 550,
        "prompt_input_tokens_stddev": 150,
        "output_tokens_mean": 150,
        "output_tokens_stddev": 50,
        "concurrency": 10,
        "request_count": 100,
        "streaming": True,
        "tokenizer": "meta-llama/Llama-3.1-8B-Instruct",
    },
    "secrets": {
        "hf_token": secret_arn,
    },
    "tooling": {"api_standard": "openai"},
}

sm_client.create_ai_workload_config(
    AIWorkloadConfigName=workload_config_name,
    AIWorkloadConfigs={"WorkloadSpec": {"Inline": json.dumps(workload_spec)}},
)
print(f"Workload config created: {workload_config_name}")
print("Parameters are IDENTICAL to Lab 1 baseline for fair comparison.")

In [ ]:
# Run benchmark on the optimized endpoint
s3_output_location = f"s3://{bucket}/benchmark-results/lab4-optimized-{timestamp}/"
benchmark_job_name = f"bench-optimized-{timestamp}"

sm_client.create_ai_benchmark_job(
    AIBenchmarkJobName=benchmark_job_name,
    BenchmarkTarget={
        "Endpoint": {
            "Identifier": optimized_endpoint_name
        }
    },
    OutputConfig={
        "S3OutputLocation": s3_output_location
    },
    AIWorkloadConfigIdentifier=workload_config_name,
    RoleArn=role,
)

print(f"Benchmark job started: {benchmark_job_name}")
print(f"Output: {s3_output_location}")
print(f"\nWaiting for completion...")

# Wait for benchmark to complete
job_response = wait_for_benchmark_job(
    job_name=benchmark_job_name,
    region=region,
    timeout_minutes=30,
)

In [ ]:
# Parse optimized benchmark results
optimized_results = parse_benchmark_results(s3_output_location, region=region)
optimized_metrics = extract_metrics(optimized_results)
format_metrics_table(optimized_metrics, title="Optimized Endpoint - Recommended Configuration")

## 5. Side-by-Side Comparison

Now the moment of truth: how much did the recommended configuration improve over the Lab 1 baseline?

In [ ]:
# Load Lab 1 baseline metrics
try:
    with open("../results/lab1_baseline_metrics.json", "r") as f:
        baseline_metrics = json.load(f)
    print("Lab 1 baseline metrics loaded successfully!")
except FileNotFoundError:
    print("WARNING: Lab 1 baseline metrics not found.")
    print("Please run Lab 1 first, or manually enter baseline values below.")
    baseline_metrics = {}

In [ ]:
# Print side-by-side comparison
if baseline_metrics and optimized_metrics:
    print_comparison_table(
        results_list=[baseline_metrics, optimized_metrics],
        labels=["Lab 1 Baseline (JumpStart default)", "Lab 4 Optimized (Recommended)"],
        title="BASELINE vs. OPTIMIZED - Head-to-Head Comparison",
    )
else:
    print("Cannot compare - missing baseline or optimized metrics.")
    print("Showing optimized metrics only:")
    format_metrics_table(optimized_metrics, title="Optimized Endpoint Results")

## 6. Cost-Per-Token Analysis

Performance improvements are only half the story. Let's calculate the cost efficiency of each configuration.

> 💡 **Note**: On-demand pricing as of 2026. Check current pricing at https://aws.amazon.com/sagemaker/pricing/

In [ ]:
# Instance pricing (on-demand, per hour) - update as needed
INSTANCE_PRICING = {
    "ml.g5.12xlarge": 7.09,
    "ml.g5.12xlarge": 7.09,
    "ml.g6e.12xlarge": 8.50,
    "ml.p4d.24xlarge": 37.69,
}

# Get the instance types
baseline_instance = "ml.g5.12xlarge"  # What we deployed in Lab 1
optimized_instance = deploy_config.get("InstanceType", "ml.g5.12xlarge")
optimized_instance_count = deploy_config.get("InstanceCount", 1)

# Calculate costs
baseline_cost_per_hour = INSTANCE_PRICING.get(baseline_instance, 7.09)
optimized_cost_per_hour = INSTANCE_PRICING.get(optimized_instance, 7.09) * optimized_instance_count

# Get throughput values
baseline_throughput = baseline_metrics.get("throughput_tokens_per_sec", 0)
optimized_throughput = optimized_metrics.get("throughput_tokens_per_sec", 0)

# Calculate cost per million tokens
if baseline_throughput > 0:
    baseline_tokens_per_hour = baseline_throughput * 3600
    baseline_cost_per_million = (baseline_cost_per_hour / baseline_tokens_per_hour) * 1_000_000
else:
    baseline_cost_per_million = float('inf')

if optimized_throughput > 0:
    optimized_tokens_per_hour = optimized_throughput * 3600
    optimized_cost_per_million = (optimized_cost_per_hour / optimized_tokens_per_hour) * 1_000_000
else:
    optimized_cost_per_million = float('inf')

# Print cost analysis
print("=" * 70)
print("  COST-PER-TOKEN ANALYSIS")
print("=" * 70)
print(f"\n{'Metric':<35} {'Baseline':<20} {'Optimized':<20}")
print(f"{'-'*75}")
print(f"{'Instance Type':<35} {baseline_instance:<20} {optimized_instance:<20}")
print(f"{'Instance Count':<35} {'1':<20} {str(optimized_instance_count):<20}")
print(f"{'Cost/Hour':<35} ${baseline_cost_per_hour:<19.2f} ${optimized_cost_per_hour:<19.2f}")
print(f"{'Throughput (tokens/sec)':<35} {baseline_throughput:<20.1f} {optimized_throughput:<20.1f}")
print(f"{'Tokens/Hour':<35} {baseline_tokens_per_hour:<20,.0f} {optimized_tokens_per_hour:<20,.0f}")
print(f"{'Cost per 1M tokens':<35} ${baseline_cost_per_million:<19.4f} ${optimized_cost_per_million:<19.4f}")

if baseline_cost_per_million > 0 and optimized_cost_per_million > 0:
    savings_pct = ((baseline_cost_per_million - optimized_cost_per_million) / baseline_cost_per_million) * 100
    print(f"\n{'Cost Savings':<35} {savings_pct:+.1f}%")
    if savings_pct > 0:
        print(f"\n  The optimized configuration saves ${baseline_cost_per_million - optimized_cost_per_million:.4f}")
        print(f"  per million tokens generated ({savings_pct:.1f}% reduction).")

## 7. When to Use Each Approach

### Use Automated Benchmarking when:
- You already have a deployed endpoint and want to validate performance
- You want to test how your endpoint handles different traffic patterns
- You're doing regression testing after a container or framework update
- You need a quick performance check (3-5 minutes)

### Use Inference Recommendations when:
- You're deploying a new model and want the optimal configuration from the start
- You want to compare across multiple instance types without manual testing
- You want automatic optimizations (speculative decoding, kernel tuning)
- You're willing to wait 30-90 minutes for comprehensive results
- You want deployment-ready configurations (not just metrics)

### Use both together when:
- You deployed with Recommendations, then want to validate with Benchmarking under different workloads
- You want to track performance over time (periodic benchmark jobs)
- You're evaluating whether a new recommendation would improve over your current production deployment

## 8. Full Cleanup

> ⚠️ **Important**: This cleans up ALL resources from the entire workshop. Only run this when you're completely done.

Resources to delete:
- Optimized endpoint (this lab)
- Any remaining Lab 1/2 endpoints
- All workload configurations
- Secrets Manager secrets
- S3 benchmark results (optional - you may want to keep these for reference)

In [ ]:
# Delete the optimized endpoint and its resources
print("Cleaning up Lab 4 resources...")
print("-" * 40)

# Delete endpoint
sm_client.delete_endpoint(EndpointName=optimized_endpoint_name)
print(f"  Deleted endpoint: {optimized_endpoint_name}")

# Delete endpoint config
sm_client.delete_endpoint_config(EndpointConfigName=optimized_endpoint_config_name)
print(f"  Deleted endpoint config: {optimized_endpoint_config_name}")

# Delete model
sm_client.delete_model(ModelName=optimized_model_name)
print(f"  Deleted model: {optimized_model_name}")

# Delete workload config
sm_client.delete_ai_workload_config(AIWorkloadConfigName=workload_config_name)
print(f"  Deleted workload config: {workload_config_name}")

# Delete secret
secretsmanager_client.delete_secret(SecretId=secret_name, ForceDeleteWithoutRecovery=True)
print(f"  Deleted secret: {secret_name}")

print("\nLab 4 cleanup complete!")

In [ ]:
# Optional: Clean up S3 benchmark results
# Uncomment to delete all benchmark output from this workshop

# import subprocess
# subprocess.run(
#     f"aws s3 rm s3://{bucket}/benchmark-results/ --recursive",
#     shell=True
# )
# subprocess.run(
#     f"aws s3 rm s3://{bucket}/recommendation-results/ --recursive",
#     shell=True
# )
# subprocess.run(
#     f"aws s3 rm s3://{bucket}/models/llama-3.1-8b-instruct/ --recursive",
#     shell=True
# )
# print("S3 artifacts cleaned up.")

---

## Workshop Complete! 

### What you accomplished:

| Lab | What You Did | Key Learning |
|-----|-------------|--------------|
| **Lab 1** | Deployed and benchmarked a model | End-to-end benchmarking workflow |
| **Lab 2** | Explored workload parameter impact | Performance tuning intuition |
| **Lab 3** | Used Inference Recommendations | Automatic optimization and multi-instance comparison |
| **Lab 4** | Deployed recommendation and compared | Quantified improvement over baseline |

### Key takeaways:

1. **Automated Benchmarking** eliminates manual load testing - get statistically rigorous LLM metrics in minutes
2. **Workload parameters matter** - the same model performs very differently under different traffic patterns
3. **Inference Recommendations** can significantly improve performance through automatic optimizations
4. **Cost-per-token** is the right metric for comparing configurations (not just raw performance)
5. **The two services are complementary** - Recommendations finds optimal configs, Benchmarking validates them

### Next steps for production:

- Set up periodic benchmark jobs to track performance regression
- Use Recommendations whenever deploying a new model or model version
- Re-run Recommendations when new instance types become available
- Build dashboards from benchmark S3 output for ongoing monitoring
- Integrate benchmarking into your CI/CD pipeline for model deployments

### Resources:

- [Benchmark generative AI inference endpoints](https://docs.aws.amazon.com/sagemaker/latest/dg/generative-ai-inference-recommendations-benchmark.html)
- [Optimized generative AI inference recommendations](https://docs.aws.amazon.com/sagemaker/latest/dg/generative-ai-inference-recommendations.html)
- [Blog: Amazon SageMaker AI now supports optimized generative AI inference recommendations](https://aws.amazon.com/blogs/machine-learning/amazon-sagemaker-ai-now-supports-optimized-generative-ai-inference-recommendations/)